[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/es/lab3/lab3_parte2.ipynb)
# Práctica 3: Redes neuronales usando PyTorch Lightning
## Parte 2. Usando PyTorch Lightning

Con PyTorch Lightning podremos simplificar la implementación del bucle de entrenamiento. Lightning nos evita escribir código repetitivo para gestionar el manejo de lotes, la validación y otros aspectos que se repiten en los entrenamientos.

# Pre-requisitos

## Instalar paquetes

En esta primera parte necesitaremos `numpy`, `torch` y `lightning` (y `pandas`, `sklearn` y `seaborn` para cargar el conjunto de datos).

In [1]:
import numpy as np
import torch
import seaborn as sns
import pandas as pd
import lightning as L

np.random.seed(1234567)

ModuleNotFoundError: No module named 'lightning'

# La clase LightningModule

Para utilizar Lightning, tenemos que hacer que nuestro módulo herede de `L.LightningModule` en lugar de `nn.Module`. Además, debemos definir los siguientes métodos:
1. `training_step(self, batch, batch_idx)`, que indica cómo computar la pérdida de un lote
1. `configure_optimizers(self)`, que define qué optimizador usar

In [ ]:
import torch.nn as nn
import torch.optim as optim

tamano_entrada = 10 # El conjunto que utilizaremos tiene 10 variables
h0_size = 5
h1_size = 3

class OurLightningNetwork(L.LightningModule):
    def __init__(self):
        super().__init__()
        # TODO - Crea una única capa, que sea un Sequential como el descrito en la parte 1
        self.layer = nn.Sequential(
            nn.Linear(tamano_entrada, h0_size),
            nn.Sigmoid(),
            nn.Linear(h0_size, h1_size),
            nn.Sigmoid(),
            nn.Linear(h1_size, 1)
        )

    def forward(self, x):
        # TODO - Completar
        return self.layer(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        # TODO - Completa lo necesario para calcular la pérdida a partir de x e y. Usa la función de pérdida adecuada.
        y_pred = self(x)
        loss = fn_perdida(y_pred, y)
        # Hacemos que se muestre el valor del pérdida en cada paso del entrenamiento
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self):
        # TODO - Define un optimizador adecuado
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        return optimizer

# Instanciamos el modelo
model = OurLightningNetwork()

# Verificación: contamos parámetros entrenables
print("Número de tensores de pesos y bias:", len(list(model.parameters())))


## Cargamos el conjunto de datos

Repetimos la carga de los laboratorios anteriores.

In [ ]:
# TODO - Copia la carga de datos del notebook anterior
vectores_x, etiquetas = load_titanic()
vectores_x = torch.from_numpy(vectores_x)
etiquetas = torch.from_numpy(etiquetas)

# Lightning necesita los datos en un dataloader, así que los envolvemos con uno
from torch.utils.data import TensorDataset, DataLoader
# Dataset
dataset = TensorDataset(vectores_x, etiquetas)

# Dataloader (mezcla y divide en batches)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## Entrenamiento del modelo

En Lightning no tendremos que escribir el bucle de entrenamiento, simplemente conseguiremos un objeto `Trainer` y llamaremos a su método `fit` indicando los datos a los que queremos ajustar el modelo.



In [ ]:
# train the model (hint: here are some helpful Trainer arguments for rapid idea iteration)
trainer = L.Trainer(max_epochs=100)
trainer.fit(model=model, train_dataloaders=dataloader)